In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
# 1.加载CSV文件
df = pd.read_csv(r'D:\年后\用户增长\sample_data.csv')

# 去掉CSV末尾的空列
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
df = df.dropna(axis=1, how='all')
# 按pid去重",
df = df.drop_duplicates(subset='pid', keep='first')

#去掉无区分度的列
df = df.drop(columns=['terminal_level'], errors='ignore')


# 查看数据基本信息
print('数据形状：', df.shape)
print('\n数据前5行：')
print(df.head())
print('\n数据基本信息：')
print(df.info())
print('\n数据描述性统计：')
print(df.describe())

In [ ]:
# 2.定义连续型特征和离散型特征

# 连续型特征
fea_continuous = [
    'avg_daily_launch_cnt',       # 单日平均启动次数
    'avg_launch_interval',        # 启动间隔均值
    'std_launch_interval',        # 启动间隔标准差
    'cv_launch_interval',         # 启动间隔变异系数
    'avg_daily_play_cnt',         # 单日播放歌曲总数
    'avg_daily_valid_play_duration',  # 单日有效播放时长
    'avg_daily_total_play_duration',  # 单日播放总时长
    'valid_play_duration_ratio',  # 有效播放时长/总播放时长
    'completion_rate',            # 完播率
    'song_repeat_ratio',          # 歌曲重复度
    'dp_first_launch_ratio',      # dp拉起首次启动占比
    'top1_cp_valid_play_ratio',   # Top1 CP有效播放占比
    'cp_diversity_ratio',         # CP分散度
]

# 离散型特征
fea_discrete = [
    'is_old_version',     # 是否老版本
    'city_level',         # 城市等级
    #'age_group',          # 年龄段
    'first_launch_mode',  # 首次启动来源
]

# 对离散特征做编码（文字/类别 -> 数字）
label_encoders = {}
for col in fea_discrete:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# 用于建模的特征
features = fea_continuous + fea_discrete
X = df[features].copy()

print(f'建模特征数：{len(features)}')
print(f'样本数：{len(X)}')
X.head()

In [ ]:
# 3.训练孤立森林
# contamination：预期异常比例
# n_estimators：孤立树数量，默认100
# max_samples：每棵树最大样本数，数据量大时可设256/512
# max_features：每棵树最大特征数，特征>=10个时建议设为 sqrt(特征总数)
# random_state：随机种子

n_features = len(features)
max_feat = max(1, int(np.sqrt(n_features)))  # sqrt(18) ≈ 4

model = IsolationForest(
    contamination=0.05,
    n_estimators=200,
    max_samples=512,
    max_features=max_feat,
    random_state=42
)

df['anomaly_label'] = model.fit_predict(X)
df['anomaly_score'] = model.decision_function(X)  # 异常分数，越小越异常

In [ ]:
# 4.统计异常值数量
normal_num = (df['anomaly_label'] == 1).sum()
outlier_num = (df['anomaly_label'] == -1).sum()
print(f'正常值数量：{normal_num}')
print(f'异常值数量：{outlier_num}')
print(f'异常占比：{outlier_num / len(df) * 100:.2f}%')

# 拆分正常/异常数据
df_normal = df[df['anomaly_label'] == 1]
df_outlier = df[df['anomaly_label'] == -1]
df_outlier.head(20)

In [ ]:
# 5.异常分数分布
plt.figure(figsize=(10, 4))
sns.histplot(df['anomaly_score'], bins=50, kde=True)
plt.axvline(x=df[df['anomaly_label'] == -1]['anomaly_score'].max(), color='red', linestyle='--', label='异常阈值')
plt.title('异常分数分布')
plt.xlabel('anomaly_score（越小越异常）')
plt.legend()
plt.tight_layout()
plt.show()

- 利益：top1_cp_valid_play_ratio 与 cp_diversity_ratio
- 成本与效率：valid_play_duration_ratio
- 行为单一性：song_repeat_ratio

In [ ]:
# 6.核心特征分布对比（正常 vs 异常）
key_features = [
    'cv_launch_interval',
    'top1_cp_valid_play_ratio',
    'cp_diversity_ratio',
    'valid_play_duration_ratio',
    'song_repeat_ratio',
    'avg_daily_launch_cnt',
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, feat in enumerate(key_features):
    ax = axes[i // 3][i % 3]
    sns.histplot(df_normal[feat], kde=True, color='blue', label='正常', ax=ax, stat='density')
    sns.histplot(df_outlier[feat], kde=True, color='red', label='异常', ax=ax, stat='density')
    ax.set_title(feat)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 7.箱线图对比
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, feat in enumerate(key_features):
    ax = axes[i // 3][i % 3]
    sns.boxplot(x='anomaly_label', y=feat, data=df, ax=ax)
    ax.set_title(feat)
    ax.set_xlabel('1=正常 / -1=异常')
plt.tight_layout()
plt.show()

In [ ]:
# 8.异常值详情
print('异常值描述统计：')
print(df_outlier[features].describe())

print(f'\n异常值明细（共{len(df_outlier)}条）：')
# 按异常分数排序，最异常的排前面
df_outlier.sort_values('anomaly_score').head(30)

In [ ]:
# 9.导出异常用户列表
df_outlier[['pid', 'anomaly_score'] + features].sort_values('anomaly_score').to_csv(
    'anomaly_users_step1.csv', index=False
)
print(f'已导出 {len(df_outlier)} 条异常用户到 anomaly_users_step1.csv')

用规则精筛

In [ ]:
# 10.粗筛结果分布分析
print('粗筛异常用户关键特征分布')
print(f'总数：{len(df_outlier)}')
print()

key_cols = ['avg_daily_launch_cnt', 'avg_daily_play_cnt', 'valid_play_duration_ratio',
            'top1_cp_valid_play_ratio', 'cp_diversity_ratio', 'song_repeat_ratio',
            'dp_first_launch_ratio', 'cv_launch_interval']
print(df_outlier[key_cols].describe(percentiles=[.1, .25, .5, .75, .9]))
print()

rules = {
    '精准卡阈值刷量(valid>0.98 & play>30)': (df_outlier['valid_play_duration_ratio'] > 0.98) & (df_outlier['avg_daily_play_cnt'] > 30),
    '高频启动异常(launch>50)': df_outlier['avg_daily_launch_cnt'] > 50,
    'TOP1 CP集中刷量(top1>0.85 & repeat<0.25)': (df_outlier['top1_cp_valid_play_ratio'] > 0.85) & (df_outlier['song_repeat_ratio'] < 0.25),
    '全是DP拉起(dp=1)': df_outlier['dp_first_launch_ratio'] == 1,
    '刷量(repeat<0.2 & play>50)': (df_outlier['song_repeat_ratio'] < 0.2) & (df_outlier['avg_daily_play_cnt'] > 50),
    'TOP1 CP+卡阈值(top1>0.85 & valid>0.98)': (df_outlier['top1_cp_valid_play_ratio'] > 0.85) & (df_outlier['valid_play_duration_ratio'] > 0.98)
}

print('=== 各精筛规则命中数量 ===')
for name, mask in rules.items():
    print(f'{name}: {mask.sum()} 条')

any_rule = pd.DataFrame(rules).any(axis=1)
print(f'\n任意规则命中: {any_rule.sum()} 条')
print(f'未命中任何规则: {(~any_rule).sum()} 条')


In [ ]:
# 11.精筛：标记异常类型
df_outlier = df_outlier.copy()
df_outlier['异常类型'] = '未分类'

df_outlier.loc[df_outlier['avg_daily_launch_cnt'] > 50, '异常类型'] = '高频启动异常'
df_outlier.loc[df_outlier['dp_first_launch_ratio'] == 1, '异常类型'] = '全是DP拉起'
df_outlier.loc[
    (df_outlier['song_repeat_ratio'] < 0.2) & (df_outlier['avg_daily_play_cnt'] > 50),
    '异常类型'] = '刷量'
df_outlier.loc[
    (df_outlier['valid_play_duration_ratio'] > 0.98) & (df_outlier['avg_daily_play_cnt'] > 30),
    '异常类型'] = '精准卡有效播放阈值'
df_outlier.loc[
    (df_outlier['top1_cp_valid_play_ratio'] > 0.85) & (df_outlier['song_repeat_ratio'] < 0.25),
    '异常类型'] = 'top1CP集中刷量'
df_outlier.loc[
    (df_outlier['top1_cp_valid_play_ratio'] > 0.8) & (df_outlier['valid_play_duration_ratio'] > 0.99),
    '异常类型'] = 'CP集中+卡阈值'

print('=== 异常类型分布 ===')
print(df_outlier['异常类型'].value_counts())
print()

df_step2 = df_outlier[df_outlier['异常类型'] != '未分类'].sort_values('anomaly_score')
df_step2.to_csv('anomaly_users_step2.csv', index=False)
print(f'精筛后导出 {len(df_step2)} 条到 anomaly_users_step2.csv')

df_step2[['pid', 'anomaly_score', '异常类型', 'avg_daily_play_cnt',
           'top1_cp_valid_play_ratio', 'valid_play_duration_ratio',
           'song_repeat_ratio', 'dp_first_launch_ratio', 'avg_daily_launch_cnt']].head(20)
